# clef-evals calibration walkthrough

Three things happen in this notebook:

1. What ECE and Brier score actually measure, on a hand-built example.
2. How clef-evals judges a dataset offline (mock transport, no credentials needed).
3. Where to plug in the live run and the CI gate.

Run it from the repo root. The only extra dependency is matplotlib for the reliability diagram.

In [ ]:
import json, sys
from pathlib import Path

REPO = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO / "src"))

from clef_evals import ClefConfig, ClefClient, ClefJudge, ece, brier_score

reference = json.loads((REPO / "evals/results/published_reference.json").read_text())
print("ForecastBench Brier (lower is better):", reference["quality"]["forecastbench_brier_lower_is_better"])

## 1. What calibration means, by hand

Accuracy only asks: was the prediction right? Calibration asks a second question: when the model says 90%, is it right about 90% of the time?

- ECE bins predictions by confidence and compares, per bin, average confidence against hit rate. 0 is perfectly calibrated.
- Brier score is the mean squared error between stated confidence and the 0/1 outcome. It punishes confident mistakes harder.

In [ ]:
# Two judges, same accuracy, very different calibration.
# Both get 8 of 10 right. Judge A says 0.9 for everything it got right and everything it got wrong.
# Judge B hedges on the hard ones.
correct = [True]*8 + [False, False]
judge_a = [0.9]*10
judge_b = [0.95, 0.9, 0.85, 0.9, 0.8, 0.9, 0.85, 0.9, 0.4, 0.35]

for name, conf in [("judge A (overconfident)", judge_a), ("judge B (honest)", judge_b)]:
    print(f"{name}: accuracy=0.80  ece={ece(conf, correct):.3f}  brier={brier_score(conf, correct):.3f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def reliability_diagram(confidences, correct, n_bins=10, label=""):
    edges = np.linspace(0, 1, n_bins + 1)
    centers, accs, confs = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        idx = [i for i, c in enumerate(confidences) if lo < c <= hi or (lo == 0 and c == 0)]
        if idx:
            centers.append((lo + hi) / 2)
            accs.append(np.mean([correct[i] for i in idx]))
            confs.append(np.mean([confidences[i] for i in idx]))
    plt.bar(centers, accs, width=0.09, alpha=0.75, label=f"hit rate {label}")
    plt.plot(centers, confs, "o--", color="#D97A3D", label="stated confidence")
    plt.plot([0, 1], [0, 1], "k:", linewidth=1)
    plt.xlabel("stated confidence")
    plt.ylabel("observed accuracy")
    plt.legend()

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1); reliability_diagram(judge_a, correct, label="judge A")
plt.title(f"overconfident: ece={ece(judge_a, correct):.2f}")
plt.subplot(1, 2, 2); reliability_diagram(judge_b, correct, label="judge B")
plt.title(f"honest: ece={ece(judge_b, correct):.2f}")
plt.tight_layout(); plt.show()

## 2. Judging a dataset offline

The judge speaks to the hosted Clef API. For a dry run without credentials we wire the client to an `httpx.MockTransport` that answers with the same response shape the real API returns (`answers` + `probabilities` + `usage`). Swap the transport for the real network and nothing else changes.

In [ ]:
import httpx
from clef_evals.judge import load_eval_set

def mock_clef(request: httpx.Request) -> httpx.Response:
    """Return a well-formed Clef answer for whatever question arrives."""
    body = json.loads(request.content)
    answers = {}
    for qid, q in body["questions"].items():
        if q["type"] == "choice":
            options = list(q["criteria"])
            probs = {o: round(1/len(options), 3) for o in options}
            probs[options[0]] = round(1 - sum(probs[o] for o in options[1:]), 3)
            answers[qid] = {"type": "choice", "choice": options[0], "probabilities": probs,
                            "confidence": probs[options[0]]}
        else:
            answers[qid] = {"type": "noul", "noul": 0.5}
    return httpx.Response(200, json={"result": {
        "model": "@cf/cloudflare/clef", "answers": answers,
        "usage": {"input_tokens": 120, "output_tokens": 8}},
        "success": True, "errors": [], "messages": []})

config = ClefConfig(account_id="dry-run", api_token="not-used")
transport = httpx.MockTransport(mock_clef)
client = ClefClient(config, http_client=httpx.Client(transport=transport))
judge = ClefJudge(config, client=client)

eval_set = load_eval_set(REPO / "tests/fixtures/eval_set.json")
result = judge.evaluate(eval_set)
print(result.summary())

The mock always answers with the first option and probability 0.5, so accuracy lands near chance. That is the point: swap in the real endpoint and the same code measures the real thing.

## 3. Going live

- Live benchmark: `make eval` (runs `evals/run_eval.py` against both Clef models), needs `CLEF_ACCOUNT_ID` / `CLEF_API_TOKEN`.
- Real-API tests: `make test-integration`.
- CI gate: `clef-eval run ... --min-accuracy 0.90 --max-ece 0.15`, or the reusable action in `.github/actions/regression-gate` comparing a fresh run against a committed baseline.
- Cloud reproduction: `kaggle-kernel/` pushes this whole flow to a Kaggle CPU kernel (push/status/output loop documented in the script header).